# Поиск по всем фото из `stamps`

Для каждого снимка: детекция клейма и top-15 ближайших эталонов.

- **v3** — `weights/finetuned_v3/student_encoder.onnx` и индекс `weights/finetuned_v3/books_index.npz` (чистый Barlow, CLAHE). Конфиг `retrieval_config_v1.yaml` уже указывает на эти файлы. Колонки в таблице по-прежнему называются v1. Ч/б BT.601 и нормировка ImageNet уже внутри модели.
- **прошлый** — InfoNCE-энкодер из `configs/retrieval/retrieval_config.yaml`.

Пары «фото → номер клейма в книге» нет, поэтому class-level Top-1 и MRR по эталону здесь не из чего считать. По каждому снимку видны детекция, косинус, запас между первым и вторым местом и совпадает ли top-1 у двух энкодеров.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "pipeline.py").is_file():
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Запускайте ноутбук из репозитория CeramicNet")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.retrieve import build_detector
from src.preprocessing.crop import crop_stamp
from src.preprocessing.image import read_bgr
from src.retrieval.books import load_retrieval_config, project_path
from src.retrieval.encoder import StampEncoder
from src.retrieval.index import StampIndex

TOP_K = 15
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_colwidth", 120)


Папка `stamps` ищется рядом с репозиторием. Детектор один на оба энкодера, порог тот же, что в конфиге (`0.75`).

In [ ]:
def find_stamps(root: Path) -> Path:
    for base in (root, root.parent):
        if not base.is_dir():
            continue
        for child in sorted(base.iterdir(), key=lambda path: path.name):
            stamps = child / "stamps"
            if stamps.is_dir() and any(stamps.glob("*.png")):
                return stamps
    raise FileNotFoundError("Папка stamps не найдена рядом с репозиторием")


def load_search(config_path: Path):
    config = load_retrieval_config(config_path)
    encoder_cfg = dict(config["encoder"])
    encoder_cfg["weights"] = str(project_path(encoder_cfg["weights"]))
    encoder = StampEncoder.from_mapping(encoder_cfg)
    index = StampIndex.load(project_path(config["index"]["path"]))
    conf = float(config["confidence_config"]["conf_threshold"])
    return encoder, index, config, conf


STAMPS_DIR = find_stamps(ROOT)
paths = sorted([
    *STAMPS_DIR.glob("*.png"),
    *STAMPS_DIR.glob("*.jpg"),
    *STAMPS_DIR.glob("*.jpeg"),
])
v1_encoder, v1_index, v1_config, CONF = load_search(
    ROOT / "configs/retrieval/retrieval_config_v1.yaml"
)
old_encoder, old_index, _, _ = load_search(
    ROOT / "configs/retrieval/retrieval_config.yaml"
)
detector = build_detector(v1_config, CONF)
print(f"Фото: {len(paths)}")
print(f"Папка: {STAMPS_DIR}")
print(f"Порог детектора: {CONF}, top-{TOP_K}")


Строка таблицы — одна детекция. Если на фото клеймо не найдено, строка всё равно есть, а поля поиска пустые. В сводке ниже каждое фото голосует своей лучшей детекцией.

In [ ]:
def hit(matches, rank):
    if len(matches) < rank:
        return None
    return matches[rank - 1]


def overlap(left, right) -> int:
    return len({item["mark_id"] for item in left} & {item["mark_id"] for item in right})


def empty_row(name: str) -> dict:
    return {
        "файл": name,
        "детекций": 0,
        "номер детекции": 0,
        "score детектора": np.nan,
        "top1": np.nan,
        "top2": np.nan,
        "top3": np.nan,
        "запас": np.nan,
        "mark_id": "",
        "каталог": "",
        "книга": "",
        "завод": "",
        "top15 v1": "",
        "mark_id прошлый": "",
        "top1 прошлый": np.nan,
        "запас прошлый": np.nan,
        "top1 совпал": np.nan,
        "пересечение top3": np.nan,
    }


rows = []
cards = []
for path in tqdm(paths, desc="stamps"):
    image = read_bgr(path)
    boxes, scores = detector.predict(image)
    boxes = np.asarray(boxes, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32).reshape(-1)
    if boxes.size == 0:
        boxes = np.zeros((0, 4), dtype=np.float32)
    else:
        boxes = boxes.reshape(-1, 4)
    if len(boxes) != len(scores):
        raise RuntimeError(f"{path.name}: боксов {len(boxes)}, scores {len(scores)}")
    keep = scores >= CONF
    boxes, scores = boxes[keep], scores[keep]
    order = np.argsort(scores)[::-1]
    boxes, scores = boxes[order], scores[order]

    if len(scores) == 0:
        rows.append(empty_row(path.name))
        cards.append({"path": path, "box": None, "det_score": None, "v1": [], "old": [], "det_i": 0, "det_n": 0})
        continue

    for det_i, (box, det_score) in enumerate(zip(boxes, scores), start=1):
        crop = crop_stamp(image, box, target_size=v1_encoder.input_size)
        v1_matches = v1_index.search(v1_encoder.embed(crop), top_k=TOP_K)
        old_matches = old_index.search(old_encoder.embed(crop), top_k=TOP_K)
        first, second, third = hit(v1_matches, 1), hit(v1_matches, 2), hit(v1_matches, 3)
        old_first, old_second = hit(old_matches, 1), hit(old_matches, 2)
        top15_text = " | ".join(
            f"{item['score']:.3f} {item['mark_id']} №{item['catalog_no']}"
            for item in v1_matches
        )
        rows.append({
            "файл": path.name,
            "детекций": int(len(scores)),
            "номер детекции": det_i,
            "score детектора": float(det_score),
            "top1": None if first is None else first["score"],
            "top2": None if second is None else second["score"],
            "top3": None if third is None else third["score"],
            "запас": None if first is None or second is None else first["score"] - second["score"],
            "mark_id": "" if first is None else first["mark_id"],
            "каталог": "" if first is None else first["catalog_no"],
            "книга": "" if first is None else first["book"],
            "завод": "" if first is None else first["factory"],
            "top15 v1": top15_text,
            "mark_id прошлый": "" if old_first is None else old_first["mark_id"],
            "top1 прошлый": None if old_first is None else old_first["score"],
            "запас прошлый": (
                None if old_first is None or old_second is None
                else old_first["score"] - old_second["score"]
            ),
            "top1 совпал": (
                None if first is None or old_first is None
                else first["mark_id"] == old_first["mark_id"]
            ),
            "пересечение top3": overlap(v1_matches[:3], old_matches[:3]),
        })
        cards.append({
            "path": path,
            "box": [float(value) for value in box],
            "det_score": float(det_score),
            "v1": v1_matches,
            "old": old_matches,
            "det_i": det_i,
            "det_n": int(len(scores)),
        })

df = pd.DataFrame(rows)
report = ROOT / "reports" / "stamps_metrics_v3.csv"
report.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(report, index=False)
print(f"Строк: {len(df)}. CSV: {report}")


Сводка считается по лучшему боксу каждого фото. `запас` — насколько первое место ушло от второго. `пересечение top3` — сколько из трёх клейм v1 есть в top-3 прошлого энкодера.

In [ ]:
found = df[df["детекций"] > 0].copy()
best = (
    found.sort_values("score детектора", ascending=False)
    .groupby("файл", as_index=False)
    .head(1)
    if len(found)
    else found
)
photos = int(df["файл"].nunique())
detected_photos = int(best["файл"].nunique()) if len(best) else 0


def mean(series) -> float:
    values = pd.to_numeric(series, errors="coerce").dropna()
    if len(values) == 0:
        return float("nan")
    return float(values.mean())


top1 = pd.to_numeric(best["top1"], errors="coerce")
summary = pd.DataFrame(
    [
        ("фото", photos),
        ("фото с детекцией", detected_photos),
        ("доля фото с детекцией", detected_photos / photos if photos else float("nan")),
        ("детекции всего", int(len(found))),
        ("средний score детектора", mean(best["score детектора"])),
        ("средний top1 v1", mean(best["top1"])),
        ("медиана top1 v1", float(top1.median()) if top1.notna().any() else float("nan")),
        ("средний top1 прошлого энкодера", mean(best["top1 прошлый"])),
        ("средний запас v1", mean(best["запас"])),
        ("средний запас прошлого", mean(best["запас прошлый"])),
        ("доля совпадения top1", mean(best["top1 совпал"])),
        ("среднее пересечение top3", mean(best["пересечение top3"])),
        ("доля пар с запасом v1 < 0.05", mean(best["запас"] < 0.05)),
    ],
    columns=["метрика", "значение"],
)


def show_value(value):
    if isinstance(value, float):
        return round(value, 4)
    return value


summary["значение"] = summary["значение"].map(show_value)
display(summary)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].hist(pd.to_numeric(best["top1"], errors="coerce").dropna(), bins=20, color="#3d6f8a")
axes[0].set_title("top1 v1")
axes[0].set_xlabel("косинус")
axes[1].hist(pd.to_numeric(best["запас"], errors="coerce").dropna(), bins=20, color="#b56b3a")
axes[1].set_title("запас top1 − top2")
axes[1].set_xlabel("разница косинусов")
agree = pd.to_numeric(best["top1 совпал"], errors="coerce").dropna()
axes[2].bar(
    ["совпал", "другой"],
    [int(agree.sum()), int((1 - agree).sum())],
    color=["#3d6f8a", "#b56b3a"],
)
axes[2].set_title("top1 v1 и прошлый")
for axis in axes:
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
fig.tight_layout()
plt.show()


Полная таблица по всем фото. Если детекций несколько, у файла несколько строк.

In [ ]:
display(df.sort_values(["файл", "номер детекции"]).reset_index(drop=True))


Карточка каждого снимка: фото с рамкой, кроп и top-15 книжных клейм в два ряда. Подпись сравнивает top-1 с прошлым энкодером.

In [ ]:
from PIL import ImageFont

MATCH = 200
GAP = 8
PER_ROW = 8
PHOTO = MATCH * 2 + GAP
try:
    RANK_FONT = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial.ttf", 18)
except OSError:
    RANK_FONT = ImageFont.load_default()


def thumb_bgr(image, box, width):
    rgb = Image.fromarray(image[:, :, ::-1])
    if box is not None:
        ImageDraw.Draw(rgb).rectangle(tuple(box), outline=(210, 35, 35), width=max(4, width // 80))
    rgb.thumbnail((width, width))
    canvas = Image.new("RGB", (width, width), (244, 244, 244))
    canvas.paste(rgb, ((width - rgb.width) // 2, (width - rgb.height) // 2))
    return canvas


def book_thumb(path, width, rank):
    canvas = Image.new("RGB", (width, width), (255, 255, 255))
    if path and Path(path).is_file():
        image = Image.open(path).convert("RGB")
        image.thumbnail((width - 10, width - 10))
        canvas.paste(image, ((width - image.width) // 2, (width - image.height) // 2))
    draw = ImageDraw.Draw(canvas)
    label = str(rank)
    badge = 22 if rank < 10 else 32
    draw.rectangle((0, 0, badge, 24), fill=(255, 255, 255))
    draw.text((4, 2), label, fill=(20, 20, 20), font=RANK_FONT)
    return canvas


def match_line(item) -> str:
    if item is None:
        return "—"
    factory = item["factory"] or "—"
    return (
        f"{item['score']:.3f}  {item['mark_id']}  {item['book']}  "
        f"№{item['catalog_no']}  {factory}"
    )


for card in cards:
    image = read_bgr(card["path"])
    photo = thumb_bgr(image, card["box"], PHOTO)
    if card["box"] is None:
        crop = Image.new("RGB", (PHOTO, PHOTO), (220, 220, 220))
    else:
        crop_bgr = crop_stamp(image, np.asarray(card["box"], dtype=np.float32), target_size=224)
        crop = thumb_bgr(crop_bgr, None, PHOTO)
    shown = card["v1"][:TOP_K]
    matches = [book_thumb(item["image_path"], MATCH, item["rank"]) for item in shown]
    match_width = PER_ROW * MATCH + (PER_ROW - 1) * GAP
    width = PHOTO * 2 + GAP + match_width
    sheet = Image.new("RGB", (width, PHOTO), (244, 244, 244))
    sheet.paste(photo, (0, 0))
    sheet.paste(crop, (PHOTO + GAP, 0))
    origin = PHOTO * 2 + GAP
    for index, panel in enumerate(matches):
        row, col = divmod(index, PER_ROW)
        sheet.paste(panel, (origin + col * (MATCH + GAP), row * (MATCH + GAP)))

    if card["det_score"] is None:
        caption = f"<b>{card['path'].name}</b><br>детекции нет"
    else:
        extra = ""
        if card["det_n"] > 1:
            extra = f" · детекция {card['det_i']} из {card['det_n']}"
        lines = "<br>".join(f"{item['rank']}. {match_line(item)}" for item in shown)
        old = match_line(card["old"][0] if card["old"] else None)
        caption = (
            f"<b>{card['path'].name}</b> · детект {card['det_score']:.3f}{extra}<br>"
            f"{lines}<br>прошлый top1: {old}"
        )
    display(HTML(caption))
    display(sheet)
